MPL Data Preparation

In [118]:
# ----------------------- Import required libraries ---------------
import numpy as np
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import time

In [119]:
# ----------------------- Load dataset ----------------------
data = np.loadtxt('../data/MLoGPU_data3_train.csv', delimiter=',')
X = data[..., :-1]
y = data[..., -1]

# Fix labels (make them 0-based)
y = y - 1

print(X.shape, y.shape)

(4000, 7) (4000,)


In [120]:
# ----------------------- Train / Test split -----------------------

# Use EXACT same split settings as teammate
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)

# Verify shapes
print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

Train shape: (3000, 7) (3000,)
Test shape: (1000, 7) (1000,)


In [121]:
# ----------------------- Data normalization -----------------------

# Compute mean and std ONLY from training data
mean_train = np.mean(X_train, axis=0)
std_train = np.std(X_train, axis=0)

# Avoid division by zero
std_train[std_train == 0] = 1

# Normalize data
X_train_norm = (X_train - mean_train) / std_train
X_test_norm = (X_test - mean_train) / std_train

# Check results
print("Train mean (approx):", np.mean(X_train_norm, axis=0))
print("Train std (approx):", np.std(X_train_norm, axis=0))

Train mean (approx): [ 2.08721929e-17  3.15895458e-16 -1.23604830e-17 -1.59344760e-16
 -2.01135405e-17  4.45569507e-17  3.59305178e-16]
Train std (approx): [1. 1. 1. 1. 1. 1. 1.]


Staaaaaaaart MPL Implementation

In [122]:
# ============================
# Select device (CPU or GPU)
# ============================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

Using device: cpu


In [123]:
# ============================
# Convert data to PyTorch tensors
# ============================

# Features → float32 (VERY important for neural networks)
X_train_tensor = torch.tensor(X_train_norm, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test_norm, dtype=torch.float32)

# Labels → long (required for classification with CrossEntropyLoss)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test, dtype=torch.long)

# Print shapes
print("X_train_tensor:", X_train_tensor.shape)
print("y_train_tensor:", y_train_tensor.shape)

X_train_tensor: torch.Size([3000, 7])
y_train_tensor: torch.Size([3000])


In [124]:
# ============================
# Move data to device (CPU/GPU)
# ============================

X_train_tensor = X_train_tensor.to(device)
X_test_tensor  = X_test_tensor.to(device)

y_train_tensor = y_train_tensor.to(device)
y_test_tensor  = y_test_tensor.to(device)

print("Data moved to:", device)

Data moved to: cpu


In [125]:
# ============================
# Define MLP model
# ============================
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        
        # Input layer → hidden layer
        self.fc1 = nn.Linear(7, 32)
        
        # Activation
        self.relu = nn.ReLU()
        
        # Hidden → output layer (7 classes)
        self.fc2 = nn.Linear(32, 7)

    def forward(self, x):
        x = self.fc1(x)     # Linear transformation
        x = self.relu(x)    # Non-linearity
        x = self.fc2(x)     # Output layer
        return x

In [126]:
# ============================
# Initialize model
# ============================

model = MLP().to(device)

print(model)

MLP(
  (fc1): Linear(in_features=7, out_features=32, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=32, out_features=7, bias=True)
)


In [127]:
# ============================
# Loss function and optimizer
# ============================

# CrossEntropyLoss is used for classification
# It automatically applies Softmax internally
criterion = nn.CrossEntropyLoss()

# Adam optimizer (good default choice)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [128]:
# ============================
# Training loop
# ============================

# Number of epochs (you can increase later)
num_epochs = 50

# Store loss values (optional, useful for plotting)
loss_history = []

for epoch in range(num_epochs):
    
    # Set model to training mode
    model.train()
    
    # Forward pass: compute predictions
    outputs = model(X_train_tensor)
    
    # Compute loss
    loss = criterion(outputs, y_train_tensor)
    
    # Backward pass: compute gradients
    optimizer.zero_grad()   # Clear old gradients
    loss.backward()         # Compute new gradients
    
    # Update weights
    optimizer.step()
    
    # Save loss
    loss_history.append(loss.item())
    
    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [10/50], Loss: 1.9716
Epoch [20/50], Loss: 1.8612
Epoch [30/50], Loss: 1.7643
Epoch [40/50], Loss: 1.6794
Epoch [50/50], Loss: 1.6050


In [129]:
# ============================
# Evaluation (accuracy)
# ============================

# Set model to evaluation mode
model.eval()

# Disable gradient computation (faster)
with torch.no_grad():
    
    # Forward pass
    outputs = model(X_test_tensor)
    
    # Get predicted class (max score)
    _, predicted = torch.max(outputs, dim=1)
    
    # Compute accuracy
    correct = (predicted == y_test_tensor).sum().item()
    total = y_test_tensor.size(0)
    
    accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 41.50%


In [130]:
# ============================
# Measure training time (CPU)
# ============================
start_time = time.time()

# ---- Training loop (same as before) ----
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

end_time = time.time()

train_time = end_time - start_time

print(f"Training time (CPU): {train_time:.4f} seconds")

Training time (CPU): 0.0244 seconds


In [131]:
# ============================
# Measure inference time (CPU)
# ============================

model.eval()

start_time = time.time()

with torch.no_grad():
    outputs = model(X_test_tensor)
    _, predicted = torch.max(outputs, dim=1)

end_time = time.time()

inference_time = end_time - start_time

print(f"Inference time (CPU): {inference_time:.6f} seconds")

Inference time (CPU): 0.000810 seconds
